<a href="https://colab.research.google.com/github/AndrejHorvat1/Multibeam-Forward-Looking-Sonar/blob/main/UATD_NLM_CLAHE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import shutil
from glob import glob

augmented_base = "/content/drive/MyDrive/UATD_YOLO_Dataset_v2"
output_base    = "/content/drive/MyDrive/UATD_YOLO_Dataset_NLM_CLAHE"

for subset in ["train", "val"]:
    os.makedirs(os.path.join(output_base, f"images/{subset}"), exist_ok=True)
    os.makedirs(os.path.join(output_base, f"labels/{subset}"), exist_ok=True)

nlm_h       = 10   # jačina denoisinga po intenzitetu
nlm_hColor  = 10   # jačina denoisinga za boju (isti ili malo veći broj)
nlm_templ   = 9    # templateWindowSize
nlm_search  = 15   # searchWindowSize

clahe_clip = 2.0
clahe_grid = (8, 8)

for subset in ["train", "val"]:
    img_paths = sorted(glob(os.path.join(augmented_base, f"images/{subset}/*.jpg")))

    for img_path in img_paths:
        img_name  = os.path.basename(img_path)
        img_id    = os.path.splitext(img_name)[0]
        label_src = os.path.join(augmented_base, f"labels/{subset}/{img_id}.txt")
        label_dst = os.path.join(output_base,     f"labels/{subset}/{img_id}.txt")

        if not os.path.exists(label_src):
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"Ne mogu učitati {img_path}, preskačem…")
            continue

        denoised = cv2.fastNlMeansDenoisingColored(
            img, None,
            h=nlm_h, hColor=nlm_hColor,
            templateWindowSize=nlm_templ,
            searchWindowSize=nlm_search
        )

        lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
        L, A, B = cv2.split(lab)

        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
        L_eq = clahe.apply(L)

        lab_eq    = cv2.merge((L_eq, A, B))
        final_img = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

        out_img_path = os.path.join(output_base, f"images/{subset}/{img_name}")
        cv2.imwrite(out_img_path, final_img)

        shutil.copy(label_src, label_dst)

        print(f"{subset}/{img_name} → (NLM + CLAHE na L-kanalu) spremljeno.")

print("\nNLM + CLAHE završen")

Izlaz streaminga skraćen je na ovoliko posljednjih redaka: 5000.
train/06387.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06387_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06389.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06389_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06390.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06390_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06391.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06391_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06392.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06392_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06393.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06393_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06395.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06395_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06397.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/06397_aug.jpg → (NLM + CLAHE na L-kanalu) spremljeno.
train/0

In [ ]:
import os
import cv2
import shutil
from glob import glob

test_sets = ["test_set1", "test_set2"]

original_base = "/content/drive/MyDrive/UATD_YOLO_Dataset_Test_v2"
output_base   = "/content/drive/MyDrive/UATD_YOLO_Dataset_Test_NLM_CLAHE"

nlm_h       = 10    # jačina denoisinga (intenzitet)
nlm_hColor  = 10   # jačina denoisinga (boja)
nlm_templ   = 9   # templateWindowSize
nlm_search  = 15   # searchWindowSize

clahe_clip = 2.0
clahe_grid = (8, 8)

for set_name in test_sets:
    inp_img_dir = os.path.join(original_base, "images", set_name)
    inp_lbl_dir = os.path.join(original_base, "labels", set_name)

    out_img_dir = os.path.join(output_base, "images", set_name)
    out_lbl_dir = os.path.join(output_base, "labels", set_name)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    image_paths = sorted(glob(os.path.join(inp_img_dir, "*.jpg")))

    for img_path in image_paths:
        img_name = os.path.basename(img_path)
        img_id   = os.path.splitext(img_name)[0]
        lbl_src  = os.path.join(inp_lbl_dir, img_id + ".txt")
        if not os.path.exists(lbl_src):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        denoised = cv2.fastNlMeansDenoisingColored(
            img, None,
            h=nlm_h, hColor=nlm_hColor,
            templateWindowSize=nlm_templ,
            searchWindowSize=nlm_search
        )

        lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
        L, A, B = cv2.split(lab)

        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
        L_eq = clahe.apply(L)

        lab_eq    = cv2.merge((L_eq, A, B))
        final_img = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

        cv2.imwrite(os.path.join(out_img_dir, img_name), final_img)

        shutil.copy(lbl_src, os.path.join(out_lbl_dir, img_id + ".txt"))

        print(f"{set_name}/{img_name} → (NLM + CLAHE) spremljeno.")

print("\nNLM + CLAHE test skup gotov", output_base)

test_set1/00001.jpg → (NLM + CLAHE) spremljeno.
test_set1/00002.jpg → (NLM + CLAHE) spremljeno.
test_set1/00003.jpg → (NLM + CLAHE) spremljeno.
test_set1/00004.jpg → (NLM + CLAHE) spremljeno.
test_set1/00005.jpg → (NLM + CLAHE) spremljeno.
test_set1/00006.jpg → (NLM + CLAHE) spremljeno.
test_set1/00007.jpg → (NLM + CLAHE) spremljeno.
test_set1/00008.jpg → (NLM + CLAHE) spremljeno.
test_set1/00009.jpg → (NLM + CLAHE) spremljeno.
test_set1/00010.jpg → (NLM + CLAHE) spremljeno.
test_set1/00011.jpg → (NLM + CLAHE) spremljeno.
test_set1/00012.jpg → (NLM + CLAHE) spremljeno.
test_set1/00013.jpg → (NLM + CLAHE) spremljeno.
test_set1/00014.jpg → (NLM + CLAHE) spremljeno.
test_set1/00015.jpg → (NLM + CLAHE) spremljeno.
test_set1/00016.jpg → (NLM + CLAHE) spremljeno.
test_set1/00017.jpg → (NLM + CLAHE) spremljeno.
test_set1/00018.jpg → (NLM + CLAHE) spremljeno.
test_set1/00019.jpg → (NLM + CLAHE) spremljeno.
test_set1/00020.jpg → (NLM + CLAHE) spremljeno.
test_set1/00021.jpg → (NLM + CLAHE) spre

In [3]:
!cp -r "/content/drive/My Drive/UATD_YOLO_Dataset_NLM_CLAHE" /content/
!cp -r "/content/drive/My Drive/UATD_YOLO_Dataset_Test_NLM_CLAHE" /content/

^C
^C


In [ ]:
data_yaml = """train: /content/UATD_YOLO_Dataset_NLM_CLAHE/images/train
val: /content/UATD_YOLO_Dataset_NLM_CLAHE/images/val
nc: 10
test:
  - /content/UATD_YOLO_Dataset_Test_NLM_CLAHE/images/test_set1
  - /content/UATD_YOLO_Dataset_Test_NLM_CLAHE/images/test_set2
names: ['cube', 'ball', 'cylinder', 'human body', 'plane', 'circle cage', 'square cage', 'metal bucket', 'tyre', 'rov']
"""

os.makedirs("yolo_dataset", exist_ok=True)

with open("yolo_dataset/data.yaml", "w") as f:
    f.write(data_yaml)

In [ ]:
!pip install ultralytics --upgrade

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import shutil
import os

# Konfiguracija
DATA_YAML_PATH = "yolo_dataset/data.yaml"
BEST_MODEL_PATH = "best_yolov8.pt"
INITIAL_WEIGHTS = "yolov8s.pt"
EPOCHS = 15

# Inicijaliziraj model
model = YOLO(INITIAL_WEIGHTS)
map_scores = []
best_map = 0.0

# Loop po epohama
for epoch in range(EPOCHS):
    print(f"\n Epoha {epoch + 1}/{EPOCHS}")

    # Treniraj samo 1 epohu
    model.train(data=DATA_YAML_PATH, epochs=1, batch=8, imgsz=640, project="runs", name="custom_loop", verbose=True, exist_ok=True)

    # Validacija na test skupu
    metrics = model.val(data=DATA_YAML_PATH, split="test", imgsz=640, batch=8)

    # Dohvati mAP@0.5 i dodaj u listu
    map_50 = metrics.box.map50  # float
    print(f"mAP@0.5 test: {map_50:.4f}")
    map_scores.append(map_50)


# Plotanje rezultata
plt.plot(range(1, EPOCHS + 1), map_scores, marker='o')
plt.title("mAP@0.5 na test skupu kroz epohe")
plt.xlabel("Epoha")
plt.ylabel("mAP@0.5")
plt.grid(True)
plt.show()


!cp -r runs/custom_loop /content/drive/MyDrive/yolov8_NLM_CLAHE/